# 01 · Buscador y partición por familias de datasets

**Qué contiene.** Arma el índice de búsqueda (SQLite FTS5) sobre el catálogo y reparte los datasets en entrenamiento, validación y prueba por familias de datasets casi idénticos, para evitar fuga entre conjuntos. Las celdas 1 y 4 son dos versiones de la partición.

| | |
|---|---|
| **Runtime** | CPU (sin GPU) |
| **Entradas (`MyDrive/qlora-series-ar/data/`)** | `catalogo_verificado.csv`, `catalogo_busqueda.csv` |
| **Salidas** | `busqueda.db`, `catalogo_busqueda.csv` (con la columna `split`), `particion_datasets.csv` |
| **Costo y tiempo** | Sin costo. Pocos minutos |
| **Orden** | Después de `00b`; antes de `02` |
| **En el repositorio** | `herramientas.construir_indice`, `particion.py`, `scripts/particionar.py` (con verificación contra la partición publicada) |

**Advertencias.**
- `SAVE` guarda sobre `catalogo_busqueda.csv` y `particion_datasets.csv`. En la celda 1 se dejó en `False`; ponerlo en `True` solo después de revisar el resultado.
- El resultado publicado es el de la semilla 3479 (puntaje 0,689; series usables 5.696 / 417 / 408). La partición se calculó sobre el catálogo anterior al dataset de tipo de cambio.


In [ ]:
import os, re, shutil, sqlite3, unicodedata
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"

# 1) Cargar: series con datos + series sin datos (el agente también puede encontrarlas)
a = pd.read_csv(f"{RUTA}/catalogo_verificado.csv", low_memory=False)
b = pd.read_csv(f"{RUTA}/series_sin_datos.csv", low_memory=False)
b["estado"] = "sin_datos"
cols = ["serie_id", "dataset_id", "dataset_titulo", "dataset_tema", "distribucion_id",
        "distribucion_titulo", "serie_titulo", "serie_descripcion", "serie_unidades",
        "indice_tiempo_frecuencia", "estado"]
df = pd.concat([a[cols], b[cols]], ignore_index=True)
df["dataset_id"] = df["dataset_id"].astype(str)
assert df["serie_id"].is_unique, "Hay serie_id repetidos"
print(len(df), "series |", df["estado"].value_counts().to_dict())

# 2) Texto de búsqueda (sin acentos, sin guiones bajos, sin campos numéricos)
def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

FREQ = {"R/P1M": "mensual", "R/P3M": "trimestral", "R/P1Y": "anual",
        "R/P6M": "semestral", "R/P1D": "diaria"}
df["frecuencia"] = df["indice_tiempo_frecuencia"].map(FREQ)
df["txt_titulo"] = df["serie_titulo"].map(norm)
df["txt_contexto"] = (df["distribucion_titulo"].map(norm) + " " + df["serie_descripcion"].map(norm)
                      + " " + df["dataset_titulo"].map(norm) + " " + df["dataset_tema"].map(norm)
                      + " " + df["serie_unidades"].map(norm) + " " + df["frecuencia"].fillna(""))

# 3) Índice SQLite FTS5
DB = "/content/busqueda.db"
if os.path.exists(DB):
    os.remove(DB)
con = sqlite3.connect(DB)
try:
    con.execute("""CREATE VIRTUAL TABLE fts USING fts5(
        serie_id UNINDEXED, titulo, contexto, tokenize='unicode61 remove_diacritics 2')""")
except sqlite3.OperationalError as e:
    raise RuntimeError(f"Este SQLite no tiene FTS5: {e}")
con.executemany("INSERT INTO fts(serie_id, titulo, contexto) VALUES (?,?,?)",
                df[["serie_id", "txt_titulo", "txt_contexto"]].itertuples(index=False, name=None))
con.commit()

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}
idx = df.set_index("serie_id")

def buscar(q, k=5):
    toks = [t for t in re.findall(r"[a-z0-9]+", norm(q)) if t not in STOP and len(t) > 1]
    if not toks:
        return pd.DataFrame()
    match = " OR ".join(f'"{t}"' for t in toks)
    rows = con.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                       "WHERE fts MATCH ? ORDER BY s LIMIT ?", (match, k)).fetchall()
    out = idx.loc[[r[0] for r in rows], ["serie_titulo", "distribucion_titulo", "frecuencia", "estado"]]
    out = out.reset_index()
    out["distribucion_titulo"] = out["distribucion_titulo"].str[:60]
    return out

# 4) Prueba rápida del buscador
for q in ["tipo de cambio", "inflación", "reservas internacionales",
          "exportaciones de soja", "valor agregado bruto construcción"]:
    print(f"\n>>> {q}")
    r = buscar(q)
    print(r.to_string() if len(r) else "(sin resultados)")

# 5) Partición por dataset_id (reparto voraz hacia el split con más déficit)
tam = df.groupby("dataset_id").size()
orden = tam.sample(frac=1, random_state=42).sort_values(ascending=False, kind="stable")
OBJ = {"train": 0.8, "val": 0.1, "test": 0.1}
total, cargas, asign = tam.sum(), {s: 0 for s in OBJ}, {}
for ds, n in orden.items():
    s = max(OBJ, key=lambda k: OBJ[k] * total - cargas[k])
    asign[ds] = s
    cargas[s] += n
df["split"] = df["dataset_id"].map(asign)

print("\nPartición:")
print(df.groupby("split").agg(series=("serie_id", "size"), datasets=("dataset_id", "nunique")))
print("\nEstado por split:\n", pd.crosstab(df["split"], df["estado"]))
print("\nFrecuencia por split:\n", pd.crosstab(df["split"], df["frecuencia"]))
print("\nTema por split:\n", pd.crosstab(df["split"], df["dataset_tema"]))

# 6) Guardar en Drive
df.to_csv(f"{RUTA}/catalogo_busqueda.csv", index=False)
df[["dataset_id", "split"]].drop_duplicates().to_csv(f"{RUTA}/particion_datasets.csv", index=False)
con.close()
shutil.copy(DB, f"{RUTA}/busqueda.db")
print("\nGuardado: catalogo_busqueda.csv, particion_datasets.csv y busqueda.db")

In [ ]:
import difflib, re, shutil, unicodedata
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
SAVE = False      # primero False; si el chequeo coincide, cambiar a True
CAP = 150           # tope de series por familia al calcular el balance

# Resultado esperado (el que se revisó): puntaje, semilla y series usables por split
ESPERADO = {"puntaje": 0.689, "semilla": 3479, "train": 5696, "val": 417, "test": 408}

df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
df["dataset_id"] = df["dataset_id"].astype(str)
df["usable"] = df["estado"].isin(["normal", "api_adelantada"])

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower()).strip()

dsd = df.groupby("dataset_id").agg(
    titulo=("dataset_titulo", "first"),
    tema=("dataset_tema", lambda s: s.mode().iloc[0]),
    split_ant=("split", "first"),
    usables=("usable", "sum"))
ids = dsd.index.tolist()
tit = {d: norm(dsd.loc[d, "titulo"]) for d in ids}

# ---------- 1) Familias de datasets ----------
padre = {d: d for d in ids}

def raiz(x):
    while padre[x] != x:
        padre[x] = padre[padre[x]]
        x = padre[x]
    return x

def unir(a, b):
    padre[raiz(b)] = raiz(a)

pares = [(difflib.SequenceMatcher(None, tit[a], tit[b]).ratio(), a, b)
         for i, a in enumerate(ids) for b in ids[i + 1:]]
for s, a, b in pares:                       # regla a) títulos muy parecidos
    if s >= 0.85:
        unir(a, b)

por_frase = {}                              # regla b) misma primera frase
for d in ids:
    por_frase.setdefault(tit[d].split(". ")[0], []).append(d)
for lista in por_frase.values():
    for d in lista[1:]:
        unir(lista[0], d)

MANUAL = [["193", "194", "195", "196", "197", "198", "199", "464"],  # IPC provincias, CABA y Gran Resistencia
          ["448", "449", "450"],                                    # precios mayoristas
          ["183", "184", "185"],                                    # Balance cambiario: cobros y pagos
          ["75", "76", "77", "78", "79"],                           # Intercambio comercial argentino
          ["152", "153", "154", "157"]]                             # asalariados registrados, sector privado

for g in MANUAL:                            # regla c) uniones manuales
    for d in g[1:]:
        if d in padre and g[0] in padre:
            unir(g[0], d)

dsd["familia"] = [raiz(d) for d in ids]
df["familia"] = df["dataset_id"].map(dsd["familia"])

fam = dsd.groupby("familia").agg(datasets=("titulo", "size"), usables=("usables", "sum"))
print(f"Datasets: {len(dsd)} | familias: {len(fam)} | con más de un dataset: {(fam['datasets'] > 1).sum()}")

# ---------- 2) Reparto por familia ----------
u = df[df["usable"]]
todas = pd.Index(sorted(dsd["familia"].unique()))
tam = u.groupby("familia").size().reindex(todas).fillna(0)
factor = np.minimum(1.0, CAP / tam.clip(lower=1))
f_tema = dsd.groupby("familia")["tema"].agg(lambda s: s.mode().iloc[0]).reindex(todas)

partes = [pd.crosstab(u["familia"], u[c]).add_prefix(c + "=") for c in ["dataset_tema", "frecuencia"]]
est = (pd.crosstab(df["familia"], df["estado"])
         .reindex(columns=["sin_datos", "api_truncada"], fill_value=0).add_prefix("estado="))
M = pd.concat(partes + [est], axis=1).reindex(todas).fillna(0).mul(factor, axis=0)
Mv, tv = M.values, M.sum().values
n = np.maximum(tam.values * factor.values, 1)
tema = f_tema.values

P = np.array([0.8, 0.1, 0.1])
SPLITS = ["train", "val", "test"]

def asignar(seed):
    rng = np.random.RandomState(seed)
    a = np.zeros(len(todas), dtype=int)
    for t in np.unique(tema):
        idx = np.where(tema == t)[0]
        orden = idx[np.argsort(-n[idx] * rng.uniform(0.6, 1.4, len(idx)))]
        total_t, carga = n[idx].sum(), np.zeros(3)
        for i in orden:
            s = int(np.argmax(P * total_t - carga))
            a[i] = s
            carga[s] += n[i]
    return a

def puntaje(a):
    cnt = np.eye(3)[a].T @ Mv
    return (np.abs(cnt - P[:, None] * tv[None, :]) / np.maximum(tv, 100)[None, :]).sum()

mejor, mejor_p, mejor_seed = None, np.inf, None
for seed in range(5000):
    a = asignar(seed)
    p = puntaje(a)
    if p < mejor_p:
        mejor, mejor_p, mejor_seed = a, p, seed
print(f"\nMejor puntaje: {mejor_p:.3f} (semilla {mejor_seed})")

# ---------- 3) Resultado ----------
fam_split = dict(zip(todas, [SPLITS[i] for i in mejor]))
df["split_nuevo"] = df["familia"].map(fam_split)
u = df[df["usable"]]
usables_split = u["split_nuevo"].value_counts().to_dict()

print("\nSeries usables por split (sin tope):", usables_split)
print("Con tope por familia:", pd.Series(np.minimum(tam.values, CAP), index=todas)
      .groupby(pd.Series(fam_split)).sum().to_dict())
print("Datasets por split:", u.groupby("split_nuevo")["dataset_id"].nunique().to_dict())
for c in ["dataset_tema", "frecuencia"]:
    print(f"\n{c} (usables):\n", pd.crosstab(u["split_nuevo"], u[c]))
print("\nestado (todas las series):\n", pd.crosstab(df["split_nuevo"], df["estado"]))

# ---------- 4) Chequeo contra lo revisado ----------
coincide = (abs(mejor_p - ESPERADO["puntaje"]) < 0.002
            and mejor_seed == ESPERADO["semilla"]
            and all(usables_split.get(s) == ESPERADO[s] for s in SPLITS))
print("\nChequeo contra el resultado revisado:", "COINCIDE" if coincide else "NO COINCIDE")

# ---------- 5) Guardar ----------
if SAVE:
    if not coincide:
        raise RuntimeError("El resultado no coincide con el revisado: no se guardó nada.")
    shutil.copy(f"{RUTA}/catalogo_busqueda.csv", f"{RUTA}/catalogo_busqueda_respaldo.csv")
    df["split"] = df["split_nuevo"]
    df.drop(columns=["split_nuevo"]).to_csv(f"{RUTA}/catalogo_busqueda.csv", index=False)
    df[["dataset_id", "familia", "split"]].drop_duplicates().to_csv(f"{RUTA}/particion_datasets.csv", index=False)
    print("Guardado. Respaldo del archivo anterior: catalogo_busqueda_respaldo.csv")
else:
    print("SAVE = False: no se guardó nada")

In [ ]:
usable = df[df["estado"].isin(["normal", "api_adelantada"])]

print("Frecuencia por split (solo usables):")
print(pd.crosstab(usable["split"], usable["frecuencia"]))
print("\nTema por split (solo usables):")
print(pd.crosstab(usable["split"], usable["dataset_tema"]))
print("\nDatasets por split:")
print(usable.groupby("split")["dataset_id"].nunique())

tam = usable.groupby(["split", "dataset_id"]).size().rename("series").reset_index()
print("\nLos 3 datasets más grandes de cada split:")
print(tam.sort_values("series", ascending=False).groupby("split").head(3).sort_values(["split", "series"], ascending=[True, False]))

print("\nDatasets con series diarias o semestrales:")
print(usable[usable["frecuencia"].isin(["diaria", "semestral"])]
      .groupby(["frecuencia", "dataset_id", "split"]).size())

import difflib

# 1) ¿Qué es el dataset 145 y cuánto pesa en Precios (train)?
print(usable[usable["dataset_id"] == "145"][["dataset_tema", "dataset_titulo"]]
      .drop_duplicates().to_string())
prec = usable[(usable["split"] == "train") & (usable["dataset_tema"] == "Precios")]
print("\nPrecios en train por dataset:")
print(prec.groupby("dataset_id").size().sort_values(ascending=False).head(5))

# 2) Efecto de distintos topes de series por dataset
print("\nSeries disponibles según el tope por dataset:")
por_ds = usable.groupby(["split", "dataset_id"]).size()
for cap in [20, 30, 40, 60, 100]:
    print(f"  tope {cap}:", por_ds.clip(upper=cap).groupby("split").sum().to_dict())

# 3) Fuga entre datasets: ¿cada dataset de test tiene un "gemelo" en train?
tit = usable.drop_duplicates("dataset_id").set_index("dataset_id")[["dataset_titulo", "split"]]
tr = tit[tit["split"] == "train"]
filas = []
for d, r in tit[tit["split"] == "test"].iterrows():
    mejor = max((difflib.SequenceMatcher(None, str(r["dataset_titulo"]).lower(),
                                         str(t).lower()).ratio(), td)
                for td, t in tr["dataset_titulo"].items())
    filas.append({"ds_test": d, "titulo_test": str(r["dataset_titulo"])[:50],
                  "ds_train_parecido": mejor[1], "similitud": round(mejor[0], 2),
                  "titulo_train": str(tr.loc[mejor[1], "dataset_titulo"])[:50]})
print("\nDatasets de test con título parecido a uno de train:")
print(pd.DataFrame(filas).sort_values("similitud", ascending=False).head(10).to_string())

# 4) ¿Qué fracción de las series de test comparte título con alguna de train?
tt = set(usable[usable["split"] == "train"]["serie_titulo"])
te = usable[usable["split"] == "test"]
print("\nSeries de test cuyo serie_titulo existe en train:", f"{te['serie_titulo'].isin(tt).mean():.1%}")

In [ ]:
import difflib, re, unicodedata
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"

df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
df["dataset_id"] = df["dataset_id"].astype(str)

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower()).strip()

dsd = df.groupby("dataset_id").agg(
    titulo=("dataset_titulo", "first"),
    tema=("dataset_tema", lambda s: s.mode().iloc[0]),
    split=("split", "first"),
    series=("serie_id", "size"),
    usables=("estado", lambda s: s.isin(["normal", "api_adelantada"]).sum()))
ids = dsd.index.tolist()
tit = {d: norm(dsd.loc[d, "titulo"]) for d in ids}

# Similitud de todos los pares de datasets
pares = []
for i, a in enumerate(ids):
    for b in ids[i + 1:]:
        pares.append((difflib.SequenceMatcher(None, tit[a], tit[b]).ratio(), a, b))

# Familias: se unen los datasets con similitud >= UMBRAL
UMBRAL = 0.85
padre = {d: d for d in ids}
def raiz(x):
    while padre[x] != x:
        padre[x] = padre[padre[x]]
        x = padre[x]
    return x
for s, a, b in pares:
    if s >= UMBRAL:
        padre[raiz(b)] = raiz(a)
dsd["familia"] = [raiz(d) for d in ids]

# 1) Familias con más de un dataset
multi = dsd[dsd.groupby("familia")["familia"].transform("size") > 1]
print(f"Familias con más de un dataset (umbral {UMBRAL}):", multi["familia"].nunique())
for f, g in multi.groupby("familia"):
    print(f"\nFamilia {f}: {len(g)} datasets, {g['usables'].sum()} series usables")
    for d, r in g.iterrows():
        print(f"   ds {d:>4} | {r['split']:5s} | {r['usables']:>4} usables | {str(r['titulo'])[:70]}")
    print("   Splits que toca:", sorted(g["split"].unique()))

# 2) Impacto sobre test
cruza = dsd.groupby("familia")["split"].transform("nunique") > 1
afect = dsd[(dsd["split"] == "test") & cruza]
print("\nDatasets de test en familias que cruzan splits:", len(afect),
      "| series usables de test afectadas:", afect["usables"].sum(), "de", dsd[dsd["split"] == "test"]["usables"].sum())

# 3) Pares dudosos (0.70 a 0.85) en splits distintos, para revisar a mano
print("\nPares con similitud entre 0.70 y 0.85 en splits distintos:")
dud = [(s, a, b) for s, a, b in pares
       if 0.70 <= s < UMBRAL and dsd.loc[a, "split"] != dsd.loc[b, "split"]]
for s, a, b in sorted(dud, reverse=True)[:25]:
    print(f"{s:.2f} | {a} ({dsd.loc[a, 'split']}) {str(dsd.loc[a, 'titulo'])[:45]}"
          f"  <->  {b} ({dsd.loc[b, 'split']}) {str(dsd.loc[b, 'titulo'])[:45]}")

In [ ]:
import difflib, re, shutil, unicodedata
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
SAVE = False        # ponerlo en True recién después de revisar el resultado
CAP = 150           # tope de series por familia al calcular el balance
MIN_TEST_TEMA = 30  # mínimo de series usables por tema en test

PESO_PENAL = 0.0         # 0 = sin penalización por mínimo de test
BUSQUEDA_LOCAL = False   # False = sin búsqueda local

df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
df["dataset_id"] = df["dataset_id"].astype(str)
df["usable"] = df["estado"].isin(["normal", "api_adelantada"])

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower()).strip()

dsd = df.groupby("dataset_id").agg(
    titulo=("dataset_titulo", "first"),
    tema=("dataset_tema", lambda s: s.mode().iloc[0]),
    split_ant=("split", "first"),
    usables=("usable", "sum"))
ids = dsd.index.tolist()
tit = {d: norm(dsd.loc[d, "titulo"]) for d in ids}

# ---------- 1) Familias de datasets ----------
padre = {d: d for d in ids}

def raiz(x):
    while padre[x] != x:
        padre[x] = padre[padre[x]]
        x = padre[x]
    return x

def unir(a, b):
    padre[raiz(b)] = raiz(a)

pares = [(difflib.SequenceMatcher(None, tit[a], tit[b]).ratio(), a, b)
         for i, a in enumerate(ids) for b in ids[i + 1:]]
for s, a, b in pares:                       # regla a) títulos muy parecidos
    if s >= 0.85:
        unir(a, b)

por_frase = {}                              # regla b) misma primera frase
for d in ids:
    por_frase.setdefault(tit[d].split(". ")[0], []).append(d)
for lista in por_frase.values():
    for d in lista[1:]:
        unir(lista[0], d)

MANUAL = [["193", "194", "195", "196", "197", "198", "199", "464"],  # IPC provincias, CABA y Gran Resistencia
          ["448", "449", "450"],                                    # precios mayoristas
          ["183", "184", "185"],                                    # Balance cambiario: cobros y pagos
          ["75", "76", "77", "78", "79"],                           # Intercambio comercial argentino
          ["152", "153", "154", "157"]]                             # asalariados registrados, sector privado

for g in MANUAL:                            # regla c) uniones manuales
    for d in g[1:]:
        if d in padre and g[0] in padre:
            unir(g[0], d)

dsd["familia"] = [raiz(d) for d in ids]
df["familia"] = df["dataset_id"].map(dsd["familia"])

fam = dsd.groupby("familia").agg(datasets=("titulo", "size"), usables=("usables", "sum"),
                                 splits_ant=("split_ant", lambda s: sorted(set(s))))
multi = fam[fam["datasets"] > 1].sort_values("usables", ascending=False)
print(f"Datasets: {len(dsd)} | familias: {len(fam)} | con más de un dataset: {len(multi)}")
for f, r in multi.iterrows():
    g = dsd[dsd["familia"] == f]
    print(f"\nFamilia {f}: {r['datasets']} datasets, {r['usables']} usables | split anterior: {r['splits_ant']}")
    print("   ids:", list(g.index), "|", str(g["titulo"].iloc[0])[:60])

print("\nPares NO unidos con similitud >= 0.74 (revisar a mano):")
rest = [(s, a, b) for s, a, b in pares if s >= 0.74 and dsd.loc[a, "familia"] != dsd.loc[b, "familia"]]
for s, a, b in sorted(rest, reverse=True)[:12]:
    print(f"{s:.2f} | {a} {str(dsd.loc[a, 'titulo'])[:42]}  <->  {b} {str(dsd.loc[b, 'titulo'])[:42]}")

# ---------- 2) Reparto por familia ----------
u = df[df["usable"]]
todas = pd.Index(sorted(dsd["familia"].unique()))
tam = u.groupby("familia").size().reindex(todas).fillna(0)
factor = np.minimum(1.0, CAP / tam.clip(lower=1))
f_tema = dsd.groupby("familia")["tema"].agg(lambda s: s.mode().iloc[0]).reindex(todas)

partes = [pd.crosstab(u["familia"], u[c]).add_prefix(c + "=") for c in ["dataset_tema", "frecuencia"]]
est = (pd.crosstab(df["familia"], df["estado"])
         .reindex(columns=["sin_datos", "api_truncada"], fill_value=0).add_prefix("estado="))
M = pd.concat(partes + [est], axis=1).reindex(todas).fillna(0).mul(factor, axis=0)
Mv, tv = M.values, M.sum().values
n = np.maximum(tam.values * factor.values, 1)
tema = f_tema.values

R = pd.crosstab(u["familia"], u["dataset_tema"]).reindex(todas).fillna(0)
Rv = R.values          # series usables reales por familia y tema, sin tope

P = np.array([0.8, 0.1, 0.1])
SPLITS = ["train", "val", "test"]

def asignar(seed):
    rng = np.random.RandomState(seed)
    a = np.zeros(len(todas), dtype=int)
    for t in np.unique(tema):
        idx = np.where(tema == t)[0]
        orden = idx[np.argsort(-n[idx] * rng.uniform(0.6, 1.4, len(idx)))]
        total_t, carga = n[idx].sum(), np.zeros(3)
        for i in orden:
            s = int(np.argmax(P * total_t - carga))
            a[i] = s
            carga[s] += n[i]
    return a

def puntaje(a):
    onehot = np.eye(3)[a].T                       # filas: train, val, test
    cnt = onehot @ Mv
    base = (np.abs(cnt - P[:, None] * tv[None, :]) / np.maximum(tv, 100)[None, :]).sum()
    test_tema = onehot[2] @ Rv                    # usables reales por tema en test
    penal = np.maximum(0, MIN_TEST_TEMA - test_tema).sum() / MIN_TEST_TEMA
    return base + 2.0 * penal

def tabla_precios(asig):
    t = u[u["dataset_tema"] == "Precios"].groupby("familia").size().rename("usables").to_frame()
    t["split"] = [SPLITS[asig[todas.get_loc(f)]] for f in t.index]
    return t.sort_values("usables", ascending=False).head(8)

# 2a) Búsqueda aleatoria
mejor, mejor_p, mejor_seed = None, np.inf, None
for seed in range(5000):
    a = asignar(seed)
    p = puntaje(a)
    if p < mejor_p:
        mejor, mejor_p, mejor_seed = a, p, seed

print("\nFamilias de Precios ANTES de la búsqueda local:")
print(tabla_precios(mejor))

# 2b) Búsqueda local: mover una familia a la vez mientras mejore el puntaje
a, p = mejor.copy(), puntaje(mejor)
mejoro = True
while mejoro:
    mejoro = False
    for i in range(len(todas)):
        for s in range(3):
            if s == a[i]:
                continue
            b = a.copy()
            b[i] = s
            pb = puntaje(b)
            if pb < p - 1e-9:
                a, p, mejoro = b, pb, True
mejor, mejor_p = a, p

print("\nFamilias de Precios DESPUÉS de la búsqueda local:")
print(tabla_precios(mejor))
print(f"\nMejor puntaje: {mejor_p:.3f} (semilla de la búsqueda aleatoria: {mejor_seed})")

# ---------- 3) Resultado ----------
fam_split = dict(zip(todas, [SPLITS[i] for i in mejor]))
df["split_nuevo"] = df["familia"].map(fam_split)
u = df[df["usable"]]

print("\nSeries usables por split (sin tope):\n", u["split_nuevo"].value_counts().to_dict())
print("Con tope por familia:", pd.Series(np.minimum(tam.values, CAP), index=todas)
      .groupby(pd.Series(fam_split)).sum().to_dict())
print("Datasets por split:", u.groupby("split_nuevo")["dataset_id"].nunique().to_dict())
for c in ["dataset_tema", "frecuencia"]:
    print(f"\n{c} (usables):\n", pd.crosstab(u["split_nuevo"], u[c]))
print("\nestado (todas las series):\n", pd.crosstab(df["split_nuevo"], df["estado"]))
print("\nLas 2 familias más grandes de cada split:")
print(u.groupby(["split_nuevo", "familia"]).size().rename("series").reset_index()
        .sort_values("series", ascending=False).groupby("split_nuevo").head(2)
        .sort_values(["split_nuevo", "series"], ascending=[True, False]).to_string(index=False))
print("\nSeries que cambian de split respecto del reparto anterior:",
      f"{(df['split'] != df['split_nuevo']).mean():.1%}")

# ---------- 4) Guardar (solo con SAVE = True) ----------
if SAVE:
    shutil.copy(f"{RUTA}/catalogo_busqueda.csv", f"{RUTA}/catalogo_busqueda_particion_por_dataset.csv")
    df["split"] = df["split_nuevo"]
    df.drop(columns=["split_nuevo"]).to_csv(f"{RUTA}/catalogo_busqueda.csv", index=False)
    df[["dataset_id", "familia", "split"]].drop_duplicates().to_csv(f"{RUTA}/particion_datasets.csv", index=False)
    print("\nGuardado (copia del reparto anterior en catalogo_busqueda_particion_por_dataset.csv)")
else:
    print("\nSAVE = False: no se guardó nada")